# Sample from Dew's text-to-image model

The landing page's model: a 176M-parameter hybrid DiT that draws 256×256 images from CLIP text, trained with Dew. Run it on a Colab GPU (Runtime → Change runtime type → a GPU) and change anything below.

In [ ]:
%pip install -q "dewml[cuda13]"

In [ ]:
from PIL import Image

from dew.artifacts import uint8_pixels
from dew.sampling import CFG, DPMSolverMultistep, EulerAncestral, Heun, TextToImage

pipe = TextToImage.from_pretrained("dewml/hybrid-dit-176m", revision="0964f57387afc938927b1047f19ed32b63fe0619")

`pipe` takes a list of prompts. `steps`, `sampler` and `guidance` choose how the sampler walks from noise to an image; `seed` fixes the noise.

In [ ]:
prompts = ["a red fox in a snowy forest", "a lighthouse at sunset, oil painting",
           "a colorful hot air balloon over a green valley", "a stained glass window with geometric patterns"]
result = pipe(prompts, key=0, steps=20, sampler=DPMSolverMultistep(), guidance=CFG(5.0))
Image.fromarray(uint8_pixels(result.host().images).transpose(1, 0, 2, 3).reshape(256, -1, 3))

The same prompts with Heun's second-order sampler over 40 steps, which evaluates the model twice a step:

In [ ]:
result = pipe(prompts, key=0, steps=40, sampler=Heun(), guidance=CFG(5.0))
Image.fromarray(uint8_pixels(result.host().images).transpose(1, 0, 2, 3).reshape(256, -1, 3))